### Check inkroot

* [#39](https://github.com/salgo60/Anslagstavla/issues/39)
* Notebook [Check linkroot.ipynb](https://github.com/salgo60/Anslagstavla/blob/main/Notebook/Check%20linkroot.ipynb)

In [1]:
# Kommunernas Anslagstavlor
#
import urllib3
import sys
from datetime import datetime
import pandas as pd

http = urllib3.PoolManager()
csv_url = "https://gist.githubusercontent.com/salgo60/3794c5f9eb5e81bb0f1cc9dd164ac1d7/raw/ea579f247971b5e73df9e7af416532a2daed8522/anslagstavla.csv"



def get_results(csv_url):
    print(csv_url)
    df = pd.read_csv(csv_url)
    #df = pd.read_csv(url)
#    print(df.head(10))
    return df

def check(url, wikibase, label):
    try:
        print ("\tNext", url) 
        r = http.request(
                "GET",
                url,
                preload_content=False,
                timeout=urllib3.Timeout(connect=5, read=10),
                redirect=True,
                headers={"User-Agent": "Mozilla/5.0"}
            )
        r.release_conn()
        content = r.data.decode("utf-8", errors="ignore")

        if r.status != 200:
            print(
                f"Status {r.status} | {url} | {wikibase} | {label}"
            )
            return False
        if "underhåll" in content:
            print("\t\t UNDERHÅLL", url)
            return False
        if "Tillfälligt stängd för underhåll" in content:
            print("\t\t\t UNDERHÅLL", url)
            return False

        if "Service Unavailable" in content:
            print("503-INNEHÅLL", url)
            return False
        return True
    except urllib3.exceptions.ReadTimeoutError as e:
        print("TIMEOUT", url)
        print(type(e).__name__, e)
        print(str(e))
        return False
    except Exception as e:
        print(f"ERROR: {url}")
        print(type(e).__name__, e)
        print(str(e))
        return False

start_time  = datetime.now()
print("Last run: ", start_time)
results = get_results(csv_url)
print ("Number records: " + str(len(results)))
ok = 0
notok = 0
results = results.reset_index()
for index, row in results.iterrows():
    #print(row)
    try:
        currentURL = row["oldurl"]
        wd = row["r"]
        wikibase = wd.replace("https://sweopendata.wikibase.cloud/entity/","")
        wdLabel = row["rLabel"]
        if "www.nora.se" in currentURL:
            print("Skip Nora")
            continue
        if "staffanstorp" in currentURL:
            print("Skip staffanstorp")
            continue

        if check(currentURL,wikibase, wdLabel):
            ok += 1
        else:
            notok += 1
    except Exception as error:
        print("An error occurred: ", wd, " - ", type(error).__name__) # An error occurred: NameError

procent = notok / (ok + notok) * 100
print("OK: ",ok,"\t not ok",notok,"\t procent problem", procent)



Last run:  2026-10-21 19:44:35.881327
https://gist.githubusercontent.com/salgo60/3794c5f9eb5e81bb0f1cc9dd164ac1d7/raw/ea579f247971b5e73df9e7af416532a2daed8522/anslagstavla.csv
Number records: 293
	Next https://www.herjedalen.se/organisation-och-politik/anslagstavla.html
Status 404 | https://www.herjedalen.se/organisation-och-politik/anslagstavla.html | Q241 | Härjedalens kommunala Anslagstavla
	Next https://www.kavlinge.se/kommun-och-politik/digital-anslagstavla
		 UNDERHÅLL https://www.kavlinge.se/kommun-och-politik/digital-anslagstavla
	Next https://www.jokkmokk.se/kommun-samhalle/kommun-och-Politik/politik-och-delaktighet/anslagstavla/
	Next https://www.hultsfred.se/artikel/anslagstavla/
Status 403 | https://www.hultsfred.se/artikel/anslagstavla/ | Q245 | Hultfreds kommuns Digitala Anslagstavla
	Next https://www.gotene.se/anslagstavla
	Next https://www.hogsby.se/Kommun-och-politik/Anslagstavla-officiell
ERROR: https://www.hogsby.se/Kommun-och-politik/Anslagstavla-officiell
MaxRetryE

In [2]:
end = datetime.now()
print("Ended: ", end)
print('Time elapsed (hh:mm:ss.ms) {}'.format(datetime.now() - start_time))
print("Last run: ", start_time)

Ended:  2026-10-21 19:52:00.321270
Time elapsed (hh:mm:ss.ms) 0:07:24.441213
Last run:  2026-10-21 19:44:35.881327


In [3]:
print("OK: ",ok,"\t not ok",notok,"\t procent problem", procent)


OK:  117 	 not ok 174 	 procent problem 59.79381443298969
